In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlalchemy

In the following cell we bind an sqlalchemy database engine to a database named 'tech_ceo_info.db' in the current directory.

In [2]:
db = sqlalchemy.create_engine('sqlite:///tech_ceo_info.db')

### Variable selection and filtering

Now we create an SQL query as a Python string, and then ask the sqlalchemy engine to apply the query to the relation (table) named 'baby' in the database. It returns a DataFrame.  Note the use of ''' ... ''' to create the multi-line query, and that the query must always be terminated with a semi-colon.

This query reads in all tuples from the gpa relation.

In [3]:
# SQL query saved in a Python string
query = ''' 
SELECT *
FROM dep_gpa_data;
'''

pd.read_sql(query, db)

,index,name,gpa,age,dept
0,0,Sergey Brin,2.8,45,CS
1,1,Danah Boyd,3.9,40,CS
2,2,Bill Gates,1.0,63,CS
3,3,Hillary Mason,4.0,39,DATASCI
4,4,Mike Olson,3.7,53,CS
5,5,Mark Zuckerburg,3.8,34,CS
6,6,Sheryl Sandberg,3.6,49,BUSINESS
7,7,Susan Wojcicki,3.8,50,BUSINESS
8,8,Marissa Mayer,2.6,43,BUSINESS


We can query any relation in our database. 

In [4]:
query = ''' 
SELECT *
FROM tech_ceo_net_worth;
'''

pd.read_sql(query, db)

,index,name,net_worth_millions
0,0,Sergey Brin,82200.0
1,1,Danah Boyd,6.0
2,2,Bill Gates,1100.0
3,3,Hillary Mason,NaN
4,4,Mike Olson,61.0
5,5,Mark Zuckerburg,48400.0
6,6,Sheryl Sandberg,15000.0
7,7,Susan Wojcicki,765.0
8,8,Marissa Mayer,600.0


This next query reads in just the 'Name' column from the gpa relation. Again, a DataFrame is returned.

In [5]:
query = ''' 
SELECT name
FROM dep_gpa_data;
''' 

pd.read_sql(query, db)

,name
0,Sergey Brin
1,Danah Boyd
2,Bill Gates
3,Hillary Mason
4,Mike Olson
5,Mark Zuckerburg
6,Sheryl Sandberg
7,Susan Wojcicki
8,Marissa Mayer


This query reads the 'Name' and 'Count' columns from the gpa relation.

In [6]:
query = ''' 
SELECT 
    name,
    dept
FROM dep_gpa_data;
''' 

pd.read_sql(query, db)

,name,dept
0,Sergey Brin,CS
1,Danah Boyd,CS
2,Bill Gates,CS
3,Hillary Mason,DATASCI
4,Mike Olson,CS
5,Mark Zuckerburg,CS
6,Sheryl Sandberg,BUSINESS
7,Susan Wojcicki,BUSINESS
8,Marissa Mayer,BUSINESS


The next query selects all tuples from the gpa relation where the department is BUSINESS

In [7]:
query = ''' 
SELECT *
FROM dep_gpa_data
WHERE dept = "BUSINESS";
'''

pd.read_sql(query, db)

,index,name,gpa,age,dept
0,6,Sheryl Sandberg,3.6,49,BUSINESS
1,7,Susan Wojcicki,3.8,50,BUSINESS
2,8,Marissa Mayer,2.6,43,BUSINESS


A slightly more complex constraint - want all tuples where gpa was greater than 3.5 and the department is either business or data science

In [8]:
query = ''' 
SELECT *
FROM dep_gpa_data
WHERE gpa > 3.5
  AND (dept = "BUSINESS"
       OR dept = "DATASCI");
'''

pd.read_sql(query, db)

,index,name,gpa,age,dept
0,3,Hillary Mason,4.0,39,DATASCI
1,6,Sheryl Sandberg,3.6,49,BUSINESS
2,7,Susan Wojcicki,3.8,50,BUSINESS


In this query we use the ORDER BY clause to cause the tuples to be sorted by decreasing order of the values in the 'age' column..

In [9]:
query = ''' 
SELECT *
FROM dep_gpa_data
WHERE dept = "CS"
ORDER BY age DESC
'''

pd.read_sql(query, db)

,index,name,gpa,age,dept
0,2,Bill Gates,1.0,63,CS
1,4,Mike Olson,3.7,53,CS
2,0,Sergey Brin,2.8,45,CS
3,1,Danah Boyd,3.9,40,CS
4,5,Mark Zuckerburg,3.8,34,CS


This query selects all tuples where the department was 'CS' and the gpa was less than 2.0

In [10]:
query = ''' 
SELECT *
FROM dep_gpa_data
WHERE dept = "CS"
  AND gpa < 2.0;
'''

pd.read_sql(query, db)

,index,name,gpa,age,dept
0,2,Bill Gates,1.0,63,CS


### Aggregation

Now we'll start to experiment with some aggregation functions; the next query computes the sum of all ages represented in the relation, over all departments.

In [11]:
query = ''' 
SELECT SUM(age) AS total_age
FROM dep_gpa_data
'''

pd.read_sql(query, db)

,total_age
0,416


Now we sum over all ages by department.

In [12]:
query = ''' 
SELECT dept, SUM(age)
FROM dep_gpa_data
GROUP BY dept
'''

pd.read_sql(query, db)

,dept,SUM(age)
0,BUSINESS,142
1,CS,235
2,DATASCI,39


The next query returns all names by department where gpa is better than 3

In [13]:
query = ''' 
SELECT name, dept, gpa
FROM dep_gpa_data
WHERE gpa > 3
GROUP BY dept
'''

pd.read_sql(query, db)

,name,dept,gpa
0,Sheryl Sandberg,BUSINESS,3.6
1,Danah Boyd,CS,3.9
2,Hillary Mason,DATASCI,4.0


The next query uses AVG instead of SUM, showing the mean gpa in each department.

In [14]:
query = ''' 
SELECT dept, AVG(gpa)
FROM dep_gpa_data
GROUP BY dept
'''

pd.read_sql(query, db)

,dept,AVG(gpa)
0,BUSINESS,3.333333
1,CS,3.040000
2,DATASCI,4.000000


Find all of the unique deptartments in the relation.

In [15]:
query = ''' 
SELECT DISTINCT dept
FROM dep_gpa_data
'''

pd.read_sql(query, db)

,dept
0,CS
1,DATASCI
2,BUSINESS


The next query counts the number of unique names.

In [16]:
query = ''' 
SELECT COUNT(DISTINCT dept) AS n_departments
FROM dep_gpa_data
'''

pd.read_sql(query, db)

,n_departments
0,3


In [17]:
query = ''' 
SELECT *
FROM (
    SELECT *
    FROM dep_gpa_data
    ORDER BY RANDOM()
    LIMIT 4
)
WHERE gpa > 3
'''

pd.read_sql(query, db)

,index,name,gpa,age,dept
0,1,Danah Boyd,3.9,40,CS
1,4,Mike Olson,3.7,53,CS
2,3,Hillary Mason,4.0,39,DATASCI
3,7,Susan Wojcicki,3.8,50,BUSINESS


#### NULL values

Note that we have a null value in our second relation

In [18]:
query = ''' 
SELECT *
FROM tech_ceo_net_worth
'''

pd.read_sql(query, db)

,index,name,net_worth_millions
0,0,Sergey Brin,82200.0
1,1,Danah Boyd,6.0
2,2,Bill Gates,1100.0
3,3,Hillary Mason,NaN
4,4,Mike Olson,61.0
5,5,Mark Zuckerburg,48400.0
6,6,Sheryl Sandberg,15000.0
7,7,Susan Wojcicki,765.0
8,8,Marissa Mayer,600.0


We can't use a normal predicate to filter for this null value. It does not represent an actual value. 

In [19]:
query = ''' 
SELECT *
FROM tech_ceo_net_worth
WHERE net_worth_millions = NULL
'''

pd.read_sql(query, db)

,index,name,net_worth_millions


We do have tools to filter using NULLs however.

In [20]:
query = ''' 
SELECT *
FROM tech_ceo_net_worth
WHERE net_worth_millions IS NOT NULL
'''

pd.read_sql(query, db)

,index,name,net_worth_millions
0,0,Sergey Brin,82200.0
1,1,Danah Boyd,6.0
2,2,Bill Gates,1100.0
3,4,Mike Olson,61.0
4,5,Mark Zuckerburg,48400.0
5,6,Sheryl Sandberg,15000.0
6,7,Susan Wojcicki,765.0
7,8,Marissa Mayer,600.0


Most aggregation functions ignore NULLs. 

In [21]:
query = ''' 
SELECT AVG(net_worth_millions)
FROM tech_ceo_net_worth
'''

pd.read_sql(query, db)

,AVG(net_worth_millions)
0,18516.5


### Joining

Note that unlike in 'pandas' we can't just select variables from different objects and assume they have the same index. If we try to do so, we end up with a **Cross Join**

In [22]:
query = ''' 
SELECT gpa, net_worth_millions
FROM dep_gpa_data, tech_ceo_net_worth
'''

pd.read_sql(query, db)

,gpa,net_worth_millions
0,2.8,82200.0
1,2.8,6.0
2,2.8,1100.0
3,2.8,NaN
4,2.8,61.0
...,...,...
76,2.6,61.0
77,2.6,48400.0
78,2.6,15000.0
79,2.6,765.0


To include information from both relations, we need to join.

To join we need to provide (1) the variables from both relations we want in our resulting Dataframe, (2) the type of join, and (3) the key to join on. 

In [23]:
query = ''' 
SELECT dep_gpa_data.name, dep_gpa_data.gpa, 
    tech_ceo_net_worth.net_worth_millions
FROM dep_gpa_data 
    LEFT JOIN tech_ceo_net_worth
    ON dep_gpa_data.name = tech_ceo_net_worth.name
'''

pd.read_sql(query, db)

,name,gpa,net_worth_millions
0,Sergey Brin,2.8,82200.0
1,Danah Boyd,3.9,6.0
2,Bill Gates,1.0,1100.0
3,Hillary Mason,4.0,NaN
4,Mike Olson,3.7,61.0
5,Mark Zuckerburg,3.8,48400.0
6,Sheryl Sandberg,3.6,15000.0
7,Susan Wojcicki,3.8,765.0
8,Marissa Mayer,2.6,600.0


The need to reference the full relation name when references each variable is cumbersome. To avoid we use aliases. 

In [24]:
query = ''' 
SELECT t1.name, t1.gpa, 
    t2.net_worth_millions
FROM dep_gpa_data AS t1
    LEFT JOIN tech_ceo_net_worth AS t2
    ON t1.name = t2.name
'''

pd.read_sql(query, db)

,name,gpa,net_worth_millions
0,Sergey Brin,2.8,82200.0
1,Danah Boyd,3.9,6.0
2,Bill Gates,1.0,1100.0
3,Hillary Mason,4.0,NaN
4,Mike Olson,3.7,61.0
5,Mark Zuckerburg,3.8,48400.0
6,Sheryl Sandberg,3.6,15000.0
7,Susan Wojcicki,3.8,765.0
8,Marissa Mayer,2.6,600.0


### Querying .csvs using SQL

In [32]:
import duckdb

In [33]:
pip install duckdb

Note: you may need to restart the kernel to use updated packages.


Remember the GSS data we looked at last week. We played with a much smaller subset of the data because the original .csv file was so large.

The `duckdb` package lets us use SQL to directly query .csv files without needing to keep them in working memory. Let's answer an important social question: **are cat owners or dog owners happier?**

- **HAPPY**: Happiness from 1 (happy) to 3 (unhappy)
- **CAT**: Has a cat (0 no, 1 yes)
- **DOG**: Has a dog (0 no, 1 yes)

In [34]:
query = '''
SELECT HAPPY, CAT, DOG
FROM 'C:/Users/Andrew/University of Oregon Dropbox/Andrew Muehleisen/DSCI/datasets/GSS_2019/GSS.csv'
'''

In [35]:
selection = duckdb.sql(query)
selection

IOException: IO Error: No files found that match the pattern "C:/Users/Andrew/University of Oregon Dropbox/Andrew Muehleisen/DSCI/datasets/GSS_2019/GSS.csv"

In [36]:
selection = selection.df()
selection.head()

NameError: name 'selection' is not defined

(The file is local on Andrew's machine, this part wont work if you're working through this yourself)

Always carefully consider the data types of your variables loaded in with SQL.

In [ ]:
selection.dtypes

In [ ]:
query = '''
SELECT HAPPY, CAST(CAT AS int) AS CAT, CAST(DOG AS int) AS DOG
FROM 'C:/Users/Andrew/University of Oregon Dropbox/Andrew Muehleisen/DSCI/datasets/GSS_2019/GSS.csv'
'''

In [ ]:
selection = duckdb.sql(query)
selection = selection.df()
selection.head()

In [ ]:
selection.dtypes

Now you know how to create basic SQL queries. Querying a very large data file as as if it were a database is one way in which you can work with very large datasets for your project.